# solar_efficiency_prediction

## Import and Load Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# Load data
train = pd.read_csv('dataset\\train.csv')
test = pd.read_csv('dataset\\test.csv')

In [17]:
# Basic Obsevation on data set
print(test.isnull().sum())
train.isnull().sum()

id                      0
temperature             0
irradiance              0
humidity                0
panel_age               0
maintenance_count       0
soiling_ratio           0
voltage                 0
current                 0
module_temperature      0
cloud_coverage          0
wind_speed              0
pressure                0
string_id               0
error_code              0
installation_type       0
power_output            0
temp_diff               0
irradiance_per_cloud    0
dtype: int64


id                      0
temperature             0
irradiance              0
humidity                0
panel_age               0
maintenance_count       0
soiling_ratio           0
voltage                 0
current                 0
module_temperature      0
cloud_coverage          0
wind_speed              0
pressure                0
string_id               0
error_code              0
installation_type       0
efficiency              0
power_output            0
temp_diff               0
irradiance_per_cloud    0
dtype: int64

In [60]:
train

,id,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,cloud_coverage,wind_speed,pressure,string_id,error_code,installation_type,efficiency,power_output,temp_diff,irradiance_per_cloud
0,0,7.817315,576.179270,41.24309,32.135501,4.0,0.803199,37.403527,1.963787,13.691147,62.494044,12.82491,1018.86651,0,0,2,0.562096,73.452561,5.873832,9.074540
1,1,24.785727,240.003973,1.35965,19.977460,8.0,0.479456,21.843315,0.241473,27.545096,43.851238,12.01204,1025.62385,3,0,0,0.396447,5.274577,2.759369,5.351111
2,2,46.652695,687.612799,91.26537,1.496401,4.0,0.822398,48.222882,4.191800,43.363708,49.704133,1.81440,1010.92265,2,0,2,0.573776,202.140687,-3.288987,13.561277
3,3,53.339567,735.141179,96.19096,18.491582,3.0,0.837529,46.295748,0.960567,57.720436,67.361473,8.73626,1021.84666,0,0,0,0.629009,44.470168,4.380870,10.753735
4,4,5.575374,12.241203,27.49507,30.722697,6.0,0.551833,0.000000,0.898062,6.786263,3.632000,0.52268,1008.55596,1,0,1,0.341874,0.000000,1.210889,2.642747
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,19995,16.868428,501.273896,93.53032,14.393967,3.0,0.738911,12.147711,3.005355,26.206810,1.733013,12.59412,1018.37447,1,2,2,0.664907,36.508185,9.338381,183.414363
19996,19996,53.415061,296.970303,93.98571,25.997012,2.0,0.513061,0.000000,0.532119,65.000000,64.558667,0.97699,1016.08110,3,0,1,0.354070,0.000000,11.584939,4.529840
19997,19997,2.442727,660.328019,37.96892,32.818396,9.0,0.548602,13.047950,4.075498,11.584869,57.730134,4.75094,1009.68446,3,0,2,0.419734,53.176896,9.142142,11.243428
19998,19998,25.077241,632.760700,43.01470,19.063517,4.0,0.697663,0.000000,1.068906,21.149351,78.123689,11.30416,1006.67387,0,0,2,0.661963,0.000000,-3.927890,7.997108


In [14]:
train.dtypes


id                        int64
temperature             float64
irradiance              float64
humidity                float64
panel_age               float64
maintenance_count       float64
soiling_ratio           float64
voltage                 float64
current                 float64
module_temperature      float64
cloud_coverage          float64
wind_speed              float64
pressure                float64
string_id                object
error_code               object
installation_type        object
efficiency              float64
power_output            float64
temp_diff               float64
irradiance_per_cloud    float64
dtype: object

In [11]:
train.describe()

,id,temperature,irradiance,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,cloud_coverage,efficiency,power_output,temp_diff,irradiance_per_cloud
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,9999.500000,25.077241,501.273896,17.509150,4.011450,0.698818,16.242251,1.713396,29.923807,51.294016,0.510260,29.819224,4.846566,22.774391
std,5773.647028,12.195953,244.656324,9.839019,1.950182,0.167838,17.439260,1.124438,11.825208,47.235208,0.140420,42.189697,6.198918,49.147772
min,0.000000,0.000000,-597.278646,0.001264,0.000000,0.400149,0.000000,0.000054,0.000000,0.000244,0.000000,0.000000,-102.687535,-93.762786
25%,4999.750000,17.311474,343.009829,9.238113,3.000000,0.558536,0.000000,0.813555,22.001083,26.548853,0.445613,0.000000,2.824399,5.882223
50%,9999.500000,25.077241,501.273896,17.497731,4.000000,0.697663,13.706438,1.642829,29.923807,49.704133,0.515709,14.186392,5.000000,9.886253
75%,14999.250000,32.360468,658.315046,25.832048,5.000000,0.839569,25.819050,2.410531,37.596789,73.798123,0.590324,43.081191,7.207279,18.812539
max,19999.000000,147.394168,1537.810349,34.998379,15.000000,0.999949,494.279016,7.315597,65.000000,1000.000000,0.987066,669.180635,39.922759,947.571323


## Data Cleaning

In [ ]:
# Data Cleaning
train.fillna({
    'temperature': train['temperature'].mean(),
    'irradiance': train['irradiance'].mean(),
    'panel_age': train['panel_age'].median(),
    'maintenance_count': train['maintenance_count'].median(),
    'soiling_ratio': train['soiling_ratio'].median(),
    'voltage': train['voltage'].mean(),
    'current': train['current'].mean(),
    'module_temperature': train['module_temperature'].mean(),
    'cloud_coverage': train['cloud_coverage'].median(),
    'error_code': train['error_code'].mode()[0],
    'installation_type': train['installation_type'].mode()[0],
    
}, inplace=True)

test.fillna({
    'temperature': test['temperature'].mean(),
    'irradiance': test['irradiance'].mean(),
    'panel_age': test['panel_age'].median(),
    'maintenance_count': test['maintenance_count'].median(),
    'soiling_ratio': test['soiling_ratio'].median(),
    'voltage': test['voltage'].mean(),
    'current': test['current'].mean(),
    'module_temperature': test['module_temperature'].mean(),
    'cloud_coverage': test['cloud_coverage'].median(),
    'error_code': test['error_code'].mode()[0],
    'installation_type': test['installation_type'].mode()[0]
}, inplace=True)

## Feature Engineering

In [3]:
# Feature Engineering
for df in [train, test]:
    df['power_output'] = df['voltage'] * df['current']
    df['temp_diff'] = df['module_temperature'] - df['temperature']
    df['irradiance_per_cloud'] = df['irradiance'] / (df['cloud_coverage'] + 1)
    
train['power_output'] = train['voltage'] * train['current']
train['temp_diff'] = train['module_temperature'] - train['temperature']
train['irradiance_per_cloud'] = train['irradiance'] / (train['cloud_coverage'] + 1)

    
test['power_output'] = test['voltage'] * test['current']
test['temp_diff'] = test['module_temperature'] - test['temperature']
test['irradiance_per_cloud'] = test['irradiance'] / (test['cloud_coverage'] + 1)

In [ ]:
# Check all columns for non-numeric values
for col in train.columns:
    if train[col].dtype == 'object':
        print(f"Non-numeric values in '{col}':")
        print(train[col].unique())
        
        
num_cols = train.select_dtypes(include=['float64', 'int64']).columns
cat_cols = train.select_dtypes(include='object').columns

print(cat_cols)
print(num_cols)


Non-numeric values in 'humidity':
['41.24308670850264' '1.3596482765960705' '91.26536837560256' ...
 '37.9689180401391' '43.01470184078199' '28.91878053813607']
Non-numeric values in 'wind_speed':
['12.82491203459621' '12.012043660984917' '1.814399755560454' ...
 '4.750937249871706' '11.304158443374758' '7.276421631094115']
Non-numeric values in 'pressure':
['1018.8665053152533' '1025.6238537572883' '1010.9226539809573' ...
 '1009.6844614602336' '1006.6738746072241' '1017.3948041584918']
Non-numeric values in 'string_id':
['A1' 'D4' 'C3' 'B2']
Non-numeric values in 'error_code':
['E00' 'E01' 'E02']
Non-numeric values in 'installation_type':
['tracking' 'dual-axis' 'fixed']
Index(['humidity', 'wind_speed', 'pressure', 'string_id', 'error_code',
       'installation_type'],
      dtype='object')
Index(['id', 'temperature', 'irradiance', 'panel_age', 'maintenance_count',
       'soiling_ratio', 'voltage', 'current', 'module_temperature',
       'cloud_coverage', 'efficiency', 'power_outpu

In [13]:
# Now apply convert test and train into float and  check na value.
for col in ['humidity', 'wind_speed', 'pressure']: 
    # Step 1: Convert 'humidity' column to float
    train[col] = pd.to_numeric(train[col], errors='coerce')

    # Step 2: Round the values to 4 decimal places
    train[col] = train[col].round(5)
    
    
# Now apply convert test and train into float and  check na value.
for col in ['humidity', 'wind_speed', 'pressure']: 
    # Step 1: Convert 'humidity' column to float
    test[col] = pd.to_numeric(test[col], errors='coerce')

    # Step 2: Round the values to 4 decimal places
    test[col] = test[col].round(5)
    
    

In [16]:
for col in ['humidity', 'wind_speed', 'pressure']:
    median_value = train[col].median()
    train[col].fillna(median_value, inplace=True)
    test[col].fillna(median_value, inplace=True)

C:\Users\vidha\AppData\Local\Temp\ipykernel_48764\2228389947.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna(median_value, inplace=True)
C:\Users\vidha\AppData\Local\Temp\ipykernel_48764\2228389947.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exampl

In [ ]:
# Encode categorical features
cat_cols = ['string_id', 'error_code', 'installation_type']
for col in cat_cols:
    train[col] = train[col].astype('category').cat.codes
    test[col] = test[col].astype('category').cat.codes

# Feature selection
drop_cols = ['id', 'efficiency']
X = train.drop(columns=drop_cols)
y = train['efficiency']
X_test = test.drop(columns=['id'])

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [31]:
model = CatBoostRegressor(iterations=150, learning_rate=0.1, depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)


In [32]:

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.5f}")


RMSE: 0.1061, Score: 89.39160


In [61]:
model = CatBoostRegressor(iterations=300, learning_rate=0.07, depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.1060, Score: 89.39525848


In [53]:
model = CatBoostRegressor(iterations=300, learning_rate=0.06, depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.1061, Score: 89.39311855


In [54]:
from sklearn.model_selection import GridSearchCV

params = {
    'depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1],
    'iterations': [200, 300, 500]
}

cb = CatBoostRegressor(random_seed=42, verbose=False)

grid = GridSearchCV(cb, params, cv=3, scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)


Best params: {'depth': 4, 'iterations': 500, 'learning_rate': 0.03}


In [59]:
model = CatBoostRegressor(iterations=500, learning_rate=0.03, depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.1061, Score: 89.39012126


In [62]:

from datetime import datetime
# Get current date and time
now = datetime.now()

# Format it as desired (e.g., YYYYMMDD_HHMMSS)
timestamp = now.strftime("%H%M%S")
print(f"submission_{timestamp}")

# Final prediction
preds = model.predict(X_test)
submission = pd.DataFrame({'id': test['id'], 'efficiency': preds})

submission.to_csv(f'submission_{timestamp}.csv', index=False)

submission_162733


In [63]:
train.to_csv(f'cleaned_train{timestamp}.csv', index=False)